In [ ]:
import os
from PIL import Image
from tqdm.auto import tqdm
import concurrent.futures

# --- Configuration ---
DATA_DIR = 'datasets'
# Upgraded to 512x512!
TARGET_SIZE = (512, 512) 

def resize_image(img_path):
    try:
        with Image.open(img_path) as img:
            # Convert everything to standard RGB (fixes transparent PNGs or Grayscale)
            img = img.convert('RGB')
            # Resize using a high-quality filter
            img = img.resize(TARGET_SIZE, Image.Resampling.LANCZOS)
            # Overwrite the original file
            img.save(img_path, 'JPEG', quality=90)
    except Exception as e:
        pass # Silently skip broken images

# 1. Gather all image paths
print("Scanning directory for images...")
image_paths = []
for root, _, files in os.walk(DATA_DIR):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_paths.append(os.path.join(root, file))

print(f"Found {len(image_paths)} images. Starting resize process...")

# 2. Process them in parallel using your Ryzen's 8 threads
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
    list(tqdm(executor.map(resize_image, image_paths), total=len(image_paths), desc="Resizing Images to 512x512"))

print("\n✅ Done! Your dataset is now beautifully optimized for 512x512.")